# 02 — Seleção de Variáveis

## Objetivo

Investigar quais variáveis podem contribuir para a classificação dos registros de desastres nas classes FALTA e EXCESSO.

A seleção considerará a relevância das informações, a qualidade dos dados, a redundância e o risco de vazamento da variável-alvo. As decisões serão documentadas e validadas antes do treinamento dos modelos.


### 1. Carregamento dos dados

Carregamos a planilha corrigida e definimos as classes de interesse, mantendo apenas os registros pertencentes ao escopo da classificação binária.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ARQUIVO_BASE = (
    ROOT
    / "data"
    / "raw"
    / "BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.xlsx"
)

df = pd.read_excel(
    ARQUIVO_BASE,
    sheet_name="Atlas Valores Corrigidos"
)

mapa_classes = {
    "Estiagem e Seca": "FALTA",
    "Enxurradas": "EXCESSO",
    "Inundações": "EXCESSO",
    "Alagamentos": "EXCESSO",
}

df["alvo"] = df["descricao_tipologia"].map(mapa_classes)

df_modelagem = df[df["alvo"].notna()].copy()

print("Registros para análise:", len(df_modelagem))
print("\nDistribuição das classes:")
display(df_modelagem["alvo"].value_counts())

Registros para análise: 51921

Distribuição das classes:


alvo
FALTA      33493
EXCESSO    18428
Name: count, dtype: int64

### 2. Identificação de possíveis vazamentos

Identificamos variáveis que revelam diretamente a tipologia do desastre ou podem reproduzir a definição do alvo. Essas variáveis serão investigadas para evitar que o modelo tenha acesso indireto à resposta correta.

In [2]:
variaveis_suspeitas = [
    "descricao_tipologia",
    "tipologia",
    "Cod_Cobrade",
    "Protocolo_S2iD",
    "grupo_de_desastre",
    "DH_AFETADOS_SECA_ESTIAGEM",
]

display(
    df_modelagem[variaveis_suspeitas]
    .head()
)

,descricao_tipologia,tipologia,Cod_Cobrade,Protocolo_S2iD,grupo_de_desastre,DH_AFETADOS_SECA_ESTIAGEM
0,Estiagem e Seca,4,14110,SC-D-4215406-14110-19910107,Climatológico,0
1,Estiagem e Seca,4,14110,RS-D-4313102-14110-19910112,Climatológico,0
2,Estiagem e Seca,4,14110,RS-D-4304952-14110-19910115,Climatológico,0
3,Estiagem e Seca,4,14110,SC-D-4208609-14110-19910115,Climatológico,0
4,Estiagem e Seca,4,14110,SC-D-4207650-14110-19910118,Climatológico,0


### 3. Qualidade das variáveis

Calculamos a quantidade e o percentual de valores ausentes de cada variável para identificar atributos que exigem tratamento ou cuja utilização pode não ser vantajosa.

In [3]:
resumo_variaveis = pd.DataFrame({
    "tipo": df_modelagem.dtypes.astype(str),
    "valores_ausentes": df_modelagem.isna().sum(),
    "percentual_ausente": (
        df_modelagem.isna().mean() * 100
    ).round(2),
    "valores_distintos": df_modelagem.nunique(dropna=True),
})

resumo_variaveis = resumo_variaveis.sort_values(
    "percentual_ausente",
    ascending=False
)

display(resumo_variaveis)

,tipo,valores_ausentes,percentual_ausente,valores_distintos
DA_Polui/cont do ar,str,51508,99.20,4
DA_Polui/cont do solo,str,51452,99.10,8
DA_Polui/cont da água,str,50802,97.84,8
DA_Incêndi parques/APA's/APP's,str,50467,97.20,6
DA_Dimi/exauri hídrico,str,47960,92.37,8
...,...,...,...,...
PEPR_Comércio (R$),float64,0,0.00,892
PEPR_Serviços (R$),float64,0,0.00,2735
PEPR_total_privado,float64,0,0.00,24446
PE_PLePR,float64,0,0.00,28320


### 4. Variabilidade das variáveis

Verificamos a quantidade de valores distintos para identificar variáveis constantes ou com pouca variação, que podem ter utilidade limitada para a classificação.


In [4]:
variaveis_constantes = (
    resumo_variaveis[
        resumo_variaveis["valores_distintos"] <= 1
    ]
)

print("Variáveis constantes ou sem valores observados:")
display(variaveis_constantes)

Variáveis constantes ou sem valores observados:


,tipo,valores_ausentes,percentual_ausente,valores_distintos


### 5. Lista inicial de candidatas

Organizamos as variáveis disponíveis, excluindo temporariamente a variável-alvo e os campos suspeitos de revelar diretamente sua classificação. A lista será revisada antes do pré-processamento e do treinamento.


In [5]:
colunas_excluir = [
    "alvo",
    *variaveis_suspeitas,
]

candidatas = [
    coluna
    for coluna in df_modelagem.columns
    if coluna not in colunas_excluir
]

print("Quantidade inicial de candidatas:", len(candidatas))

display(
    resumo_variaveis.loc[
        resumo_variaveis.index.intersection(candidatas)
    ]
)

Quantidade inicial de candidatas: 64


,tipo,valores_ausentes,percentual_ausente,valores_distintos
DA_Polui/cont do ar,str,51508,99.20,4
DA_Polui/cont do solo,str,51452,99.10,8
DA_Polui/cont da água,str,50802,97.84,8
DA_Incêndi parques/APA's/APP's,str,50467,97.20,6
DA_Dimi/exauri hídrico,str,47960,92.37,8
...,...,...,...,...
PEPR_Indústria (R$),float64,0,0.00,1468
PEPR_Comércio (R$),float64,0,0.00,892
PEPR_Serviços (R$),float64,0,0.00,2735
PEPR_total_privado,float64,0,0.00,24446


### 6. Variáveis candidatas

Nesta etapa, são selecionadas as variáveis candidatas à modelagem, excluindo o alvo e os campos inicialmente identificados como possíveis fontes de vazamento de dados.

In [6]:
# Variáveis com pelo menos 80% de valores ausentes
alta_ausencia = resumo_variaveis[
    resumo_variaveis["percentual_ausente"] >= 80
].copy()

print("Quantidade de variáveis com 80% ou mais de ausência:",
      len(alta_ausencia))

display(alta_ausencia)

Quantidade de variáveis com 80% ou mais de ausência: 5


,tipo,valores_ausentes,percentual_ausente,valores_distintos
DA_Polui/cont do ar,str,51508,99.20,4
DA_Polui/cont do solo,str,51452,99.10,8
DA_Polui/cont da água,str,50802,97.84,8
DA_Incêndi parques/APA's/APP's,str,50467,97.20,6
DA_Dimi/exauri hídrico,str,47960,92.37,8


### 7. Variáveis constantes

Esta etapa identifica variáveis com um único valor observado ou sem valores válidos, que não contribuem para diferenciar as classes e podem ser removidas.

In [7]:
variaveis_constantes = resumo_variaveis[
    resumo_variaveis["valores_distintos"] <= 1
]

print("Variáveis constantes ou sem valores observados:")
display(variaveis_constantes)

Variáveis constantes ou sem valores observados:


,tipo,valores_ausentes,percentual_ausente,valores_distintos


### 8. Listagem das variáveis candidatas

São exibidos os nomes das variáveis candidatas para facilitar a inspeção dos atributos e apoiar a decisão sobre quais serão utilizados na modelagem.

In [8]:
print(f"Total de variáveis candidatas: {len(candidatas)}")

for i, coluna in enumerate(candidatas, start=1):
    print(f"{i:02d}. {coluna}")

Total de variáveis candidatas: 64
01. Nome_Municipio
02. Sigla_UF
03. regiao
04. Data_Registro
05. Data_Evento
06. Cod_IBGE_Mun
07. Setores Censitários
08. Status
09. DH_Descricao
10. DH_MORTOS
11. DH_FERIDOS
12. DH_ENFERMOS
13. DH_DESABRIGADOS
14. DH_DESALOJADOS
15. DH_DESAPARECIDOS
16. DH_total_danos_humanos_diretos
17. DH_OUTROS AFETADOS
18. DM_Descricao
19. DM_Uni Habita Danificadas
20. DM_Uni Habita Destruidas
21. DM_Uni Habita Valor
22. DM_Inst Saúde Danificadas
23. DM_Inst Saúde Destruidas
24. DM_Inst Saúde Valor
25. DM_Inst Ensino Danificadas
26. DM_Inst Ensino Destruidas
27. DM_Inst Ensino Valor
28. DM_Inst Serviços Danificadas
29. DM_Inst Serviços Destruidas
30. DM_Inst Serviços Valor
31. DM_Inst Comuni Danificadas
32. DM_Inst Comuni Destruidas
33. DM_Inst Comuni Valor
34. DM_Obras de Infra Danificadas
35. DM_Obras de Infra Destruidas
36. DM_Obras de Infra Valor
37. DM_total_danos_materiais
38. DA_Descricao
39. DA_Polui/cont da água
40. DA_Polui/cont do ar
41. DA_Polui/cont d

### 9. Consolidação da análise

São reunidas as variáveis com alta proporção de valores ausentes, as variáveis constantes e os campos suspeitos de vazamento, apoiando a seleção final dos atributos.

In [9]:
print("=== VARIÁVEIS COM ALTA AUSÊNCIA ===")
display(alta_ausencia)

print("\n=== VARIÁVEIS CONSTANTES ===")
display(variaveis_constantes)

print("\n=== VARIÁVEIS SUSPEITAS DE VAZAMENTO ===")
for coluna in variaveis_suspeitas:
    if coluna in df_modelagem.columns:
        print(f"- {coluna}")

=== VARIÁVEIS COM ALTA AUSÊNCIA ===


,tipo,valores_ausentes,percentual_ausente,valores_distintos
DA_Polui/cont do ar,str,51508,99.20,4
DA_Polui/cont do solo,str,51452,99.10,8
DA_Polui/cont da água,str,50802,97.84,8
DA_Incêndi parques/APA's/APP's,str,50467,97.20,6
DA_Dimi/exauri hídrico,str,47960,92.37,8



=== VARIÁVEIS CONSTANTES ===


,tipo,valores_ausentes,percentual_ausente,valores_distintos



=== VARIÁVEIS SUSPEITAS DE VAZAMENTO ===
- descricao_tipologia
- tipologia
- Cod_Cobrade
- Protocolo_S2iD
- grupo_de_desastre
- DH_AFETADOS_SECA_ESTIAGEM


### 10. Verificacao cruzada de vazamento

Tabelas cruzadas de `grupo_de_desastre` e `DH_AFETADOS_SECA_ESTIAGEM` contra o alvo, para confirmar se essas colunas revelam a classe e devem ser excluidas.

In [10]:
# 10a. grupo_de_desastre x alvo
print("=== grupo_de_desastre x alvo ===")
ct_grupo = pd.crosstab(df_modelagem["grupo_de_desastre"], df_modelagem["alvo"])
display(ct_grupo)
print()
print("Cada grupo aparece em uma unica classe: vazamento total.")
print("grupo_de_desastre adicionada a variaveis_suspeitas.")

print()

# 10b. DH_AFETADOS_SECA_ESTIAGEM x alvo (% com valor > 0)
print("=== DH_AFETADOS_SECA_ESTIAGEM (% com valor > 0 por classe) ===")
for classe in ["FALTA", "EXCESSO"]:
    sub = df_modelagem[df_modelagem["alvo"] == classe]
    gt0 = (sub["DH_AFETADOS_SECA_ESTIAGEM"] > 0).sum()
    pct = gt0 / len(sub) * 100
    print(f"  {classe}: {gt0:,}/{len(sub):,} = {pct:.1f}%")

print()
print("A coluna so tem valores > 0 em FALTA e seu nome referencia")
print("'SECA_ESTIAGEM', que e a propria definicao da classe: vazamento.")
print("DH_AFETADOS_SECA_ESTIAGEM adicionada a variaveis_suspeitas.")

=== grupo_de_desastre x alvo ===


alvo,EXCESSO,FALTA
grupo_de_desastre,,
Climatológico,0,33493
Hidrológico,18428,0



Cada grupo aparece em uma unica classe: vazamento total.
grupo_de_desastre adicionada a variaveis_suspeitas.

=== DH_AFETADOS_SECA_ESTIAGEM (% com valor > 0 por classe) ===
  FALTA: 14,855/33,493 = 44.4%
  EXCESSO: 0/18,428 = 0.0%

A coluna so tem valores > 0 em FALTA e seu nome referencia
'SECA_ESTIAGEM', que e a propria definicao da classe: vazamento.
DH_AFETADOS_SECA_ESTIAGEM adicionada a variaveis_suspeitas.
